# 365 Probabilidades - Dia #128
## Qual a probabilidade de você chegar ao trabalho em quinze minutos?

**Tipo:** Probabilidade simples
**Data de publicação:** 2026-10-19
**Ferramenta:** Python
**Decisão analisada:** Quanto do dia vai no caminho entre a casa e o trabalho?
**Hashtag:** #365Probabilidades #Dia128

---

### 📖 A História

[Abertura da Ana. Uma ou duas linhas sobre o seu trajeto, a inserir.]

Fui ver quanto tempo o brasileiro leva para chegar ao trabalho. O Censo perguntou isso a
cada pessoa ocupada que sai de casa para trabalhar e volta no mesmo dia.

---

### 📚 O Conceito

O tempo até o trabalho é uma das poucas coisas que o Censo mede em minutos. A pergunta
é feita a quem trabalha fora de casa e volta diariamente, e a resposta cai em sete
faixas, de "até cinco minutos" a "mais de quatro horas". Com as faixas dá para
responder à pergunta do dia de frente: quantas pessoas, em cada cem, estão no trabalho
em quinze minutos.

A mesma pergunta tem respostas muito diferentes conforme o lugar. Por isso o dia desce
do Brasil para as capitais e para uma cidade do interior. E como cada faixa é uma
contagem, dá para sortear uma sala de trabalho e perguntar quantos colegas chegaram em
quinze minutos.

---

### 🧮 O Modelo

Dia de **probabilidade simples**: a pergunta vem primeiro, e o dado é contagem. As
contas são nossas.

1. **O Brasil em sete faixas.** A fatia de cada faixa e a resposta do dia, com
   intervalo.
2. **Homens e mulheres.** As mesmas faixas por sexo, e a cauda das quatro horas.
3. **Capital a capital.** Doze capitais e Bauru contra o Brasil.
4. **Uma sala de trinta.** Monte Carlo de uma sala de trabalho sorteada no Brasil, em
   São Paulo e em Bauru: quantos colegas chegaram em quinze minutos.
5. **Um ano de trajetos.** Conta de bolso: minutos por dia que a faixa representa, e
   horas por ano.

**Dados:**
- IBGE, *Censo Demográfico 2022*, resultados da amostra, deslocamento para trabalho.
  Tabela SIDRA **10331** (pessoas de 14 anos ou mais, ocupadas, que trabalham fora do
  domicílio e retornam diariamente, por tempo habitual de deslocamento e sexo) e tabela
  **10332** (as mesmas pessoas por principal meio de transporte). Consulta à API do
  SIDRA em 16/10/2026. **71.227.656 pessoas** na tabela de tempo. Brasil, doze capitais
  e Bauru (SP).
- Os resumos estão em `dia-128-tempo-deslocamento.csv` e `dia-128-meio-transporte.csv`,
  ao lado deste notebook, com as contagens exatamente como a API devolveu.

**Nota metodológica.** O fator ×0,80 do projeto não se aplica. A pergunta do Censo é
sobre um fato do dia (quanto tempo leva), e não opinião sobre si; o projeto não aplicou
o fator nos outros dias de IBGE. As faixas somam cerca de 1% a menos do que o total,
que é a parte sem tempo declarado; as fatias são calculadas sobre o total, como o IBGE
faz.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
SORTEIOS = 200_000
rng = np.random.default_rng(SEMENTE)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE = 'Fonte: IBGE, Censo 2022, SIDRA 10331 e 10332 · cálculo próprio | #365Probabilidades'

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=1):
    return vir(100 * x, casas) + "%"

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS ---
FAIXAS = ['ate_5', 'de_6_a_15', 'de_15_a_30', 'de_30_a_60', 'de_60_a_120', 'de_120_a_240', 'mais_de_240']
ROTULOS = ['até 5 min', '6 a 15 min', '15 a 30 min', '30 min a 1 h', '1 a 2 h', '2 a 4 h', 'mais de 4 h']
PONTO_MEDIO = np.array([3, 10, 22.5, 45, 90, 180, 270])   # minutos, para a conta de bolso; 4 h+ fixado em 270

tempo = pd.read_csv('dia-128-tempo-deslocamento.csv')
meio = pd.read_csv('dia-128-meio-transporte.csv')

tempo['declarado'] = tempo[FAIXAS].sum(axis=1)
tempo['ate_15'] = tempo['ate_5'] + tempo['de_6_a_15']
tempo['mais_1h'] = tempo['de_60_a_120'] + tempo['de_120_a_240'] + tempo['mais_de_240']
tempo['p15'] = tempo['ate_15'] / tempo['total']
tempo['p1h'] = tempo['mais_1h'] / tempo['total']

br = tempo[(tempo.local == 'Brasil') & (tempo.sexo == 'Total')].iloc[0]
print("=" * 72)
print("  OS DADOS · CENSO 2022, DESLOCAMENTO PARA O TRABALHO")
print("=" * 72)
print(f"\n  Pessoas ocupadas que saem para trabalhar e voltam no dia: {mil(br.total)}")
print(f"  Com tempo declarado: {mil(br.declarado)} ({pct(br.declarado / br.total)}); sem declaração: {pct(1 - br.declarado / br.total)}\n")
for f, r in zip(FAIXAS, ROTULOS):
    print(f"    {r:>14}: {mil(br[f]):>12}   {pct(br[f] / br.total):>6}")
print(f"\n  Até 15 minutos: {mil(br.ate_15)} · {pct(br.p15)}")
print(f"  Mais de 1 hora: {mil(br.mais_1h)} · {pct(br.p1h)}")

mt = meio.set_index('meio')['pessoas']
print(f"\n  Meio de transporte (tabela 10332, {mil(mt['Total'])} pessoas; universo um pouco diferente, declarado):")
for m in ['Automóvel', 'Ônibus', 'A pé', 'Motocicleta', 'Bicicleta', 'Trem ou metrô']:
    print(f"    {m:>14}: {pct(mt[m] / mt['Total'])}")
print("\n  ×0,80: não se aplica (fato do dia declarado ao Censo, não opinião).")
print("=" * 72)

In [ ]:
# --- O MODELO ---

# ── CAMADA 1 · o Brasil em sete faixas ──
post15 = stats.beta(br.ate_15 + 0.5, br.total - br.ate_15 + 0.5)
lo, hi = post15.interval(0.95)
print("=" * 72)
print("  CAMADA 1 · A RESPOSTA DO DIA")
print("=" * 72)
print(f"\n  P(chegar ao trabalho em até 15 minutos) = {pct(br.p15)}  [{pct(lo, 2)}; {pct(hi, 2)}]")
print(f"  É mais de um em cada três. Um em cada dez chega em cinco minutos ({pct(br.ate_5 / br.total)}).")
print(f"  A faixa mais comum é de 15 a 30 minutos ({pct(br.de_15_a_30 / br.total)}).")
print(f"  Mais de uma hora: {pct(br.p1h)}, uma em cada {vir(1 / br.p1h, 0)} pessoas.")
print("  O intervalo de Jeffreys trata a contagem como amostra simples; o Censo 2022 usa amostra")
print("  complexa e o IBGE não publicou o coeficiente de variação desta célula. O intervalo é a")
print("  largura mínima, não a oficial.")

# ── CAMADA 2 · homens e mulheres ──
h = tempo[(tempo.local == 'Brasil') & (tempo.sexo == 'Homens')].iloc[0]
m = tempo[(tempo.local == 'Brasil') & (tempo.sexo == 'Mulheres')].iloc[0]
print("\n" + "=" * 72)
print("  CAMADA 2 · HOMENS E MULHERES")
print("=" * 72)
print(f"\n  {'faixa':>14} {'homens':>9} {'mulheres':>9}")
for f, r in zip(FAIXAS, ROTULOS):
    print(f"  {r:>14} {pct(h[f] / h.total):>9} {pct(m[f] / m.total):>9}")
print(f"\n  Até 15 minutos: homens {pct(h.p15)} · mulheres {pct(m.p15)} · diferença {vir((m.p15 - h.p15) * 100)} pontos")
print(f"  Mais de 1 hora: homens {pct(h.p1h)} · mulheres {pct(m.p1h)} · quase iguais")
razao_cauda = (h.mais_de_240 / h.total) / (m.mais_de_240 / m.total)
print(f"  Mais de 4 horas: {mil(h.mais_de_240)} homens e {mil(m.mais_de_240)} mulheres;")
print(f"    por pessoa ocupada, {pct(h.mais_de_240 / h.total, 2)} contra {pct(m.mais_de_240 / m.total, 2)}, {vir(razao_cauda)} vezes")
# diferença até 15 min com intervalo por simulação das posteriores
d = stats.beta(m.ate_15 + .5, m.total - m.ate_15 + .5).rvs(SORTEIOS, random_state=rng) - \
    stats.beta(h.ate_15 + .5, h.total - h.ate_15 + .5).rvs(SORTEIOS, random_state=rng)
print(f"  Diferença mulheres − homens (até 15 min): {vir(d.mean() * 100, 2)} pontos [{vir(np.percentile(d, 2.5) * 100, 2)}; {vir(np.percentile(d, 97.5) * 100, 2)}]")

# ── CAMADA 3 · capital a capital ──
cap = tempo[(tempo.sexo == 'Total') & (tempo.local != 'Brasil')].sort_values('p15', ascending=False).reset_index(drop=True)
print("\n" + "=" * 72)
print("  CAMADA 3 · CAPITAL A CAPITAL (e Bauru)")
print("=" * 72)
print(f"\n  {'cidade':>20} {'até 15 min':>11} {'mais de 1 h':>12} {'pessoas':>12}")
for _, r in cap.iterrows():
    print(f"  {r.local:>20} {pct(r.p15):>11} {pct(r.p1h):>12} {mil(r.total):>12}")
sp = cap[cap.local == 'São Paulo (SP)'].iloc[0]; ba = cap[cap.local == 'Bauru (SP)'].iloc[0]
print(f"\n  Brasil {pct(br.p15)} · São Paulo {pct(sp.p15)} · Bauru {pct(ba.p15)}")
print(f"  Em São Paulo, chegar em 15 minutos é {vir(ba.p15 / sp.p15)} vezes menos comum do que em Bauru.")
print(f"  Mais de uma hora: São Paulo {pct(sp.p1h)}, Rio {pct(cap[cap.local == 'Rio de Janeiro (RJ)'].iloc[0].p1h)}, Bauru {pct(ba.p1h)}.")

# ── CAMADA 4 · uma sala de trinta ──
N_SALA = 30
print("\n" + "=" * 72)
print(f"  CAMADA 4 · UMA SALA DE {N_SALA} ({mil(SORTEIOS)} sorteios por lugar)")
print("=" * 72)
sala = {}
for nome, p in [('Brasil', br.p15), ('São Paulo', sp.p15), ('Bauru', ba.p15)]:
    x = rng.binomial(N_SALA, p, SORTEIOS)
    sala[nome] = x
    print(f"\n  {nome}: em média {vir(x.mean())} de {N_SALA} colegas chegaram em 15 minutos")
    print(f"    90% das salas entre {int(np.percentile(x, 5))} e {int(np.percentile(x, 95))} · "
          f"nenhum colega em {pct((x == 0).mean(), 2)} das salas · maioria da sala em {pct((x > N_SALA / 2).mean(), 1)}")
print("\n  Premissa: colegas sorteados ao acaso da cidade. Na vida, quem trabalha no mesmo lugar")
print("  tende a morar em bairros parecidos, o que puxa a sala para os extremos.")

# ── CAMADA 5 · um ano de trajetos (conta de bolso) ──
DIAS_ANO = 220
fat = br[FAIXAS].values.astype(float) / br.declarado
media_min = (fat * PONTO_MEDIO).sum()
print("\n" + "=" * 72)
print("  CAMADA 5 · UM ANO DE TRAJETOS (conta de bolso)")
print("=" * 72)
print(f"\n  Ponto médio de cada faixa (4 h+ fixado em 270 min) ponderado pelas fatias: {vir(media_min, 0)} minutos por trajeto")
print(f"  Ida e volta, {DIAS_ANO} dias por ano: {vir(media_min * 2 * DIAS_ANO / 60, 0)} horas por ano, "
      f"ou {vir(media_min * 2 * DIAS_ANO / 60 / 24, 1)} dias inteiros")
for nome, minutos in [('quem chega em 15 (ponto médio 10)', 10), ('quem leva de 1 a 2 h (ponto médio 90)', 90)]:
    print(f"    {nome}: {vir(minutos * 2 * DIAS_ANO / 60, 0)} horas por ano")
print("  Os 220 dias e os pontos médios são premissas nossas, declaradas.")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · o Brasil em sete faixas ══
fig1, ax1 = plt.subplots(figsize=(12, 7))
vals = [br[f] / br.total * 100 for f in FAIXAS]
cores = [VERDE, VERDE, CINZA, CINZA, VERMELHO, VERMELHO, VERMELHO]
bars = ax1.bar(ROTULOS, vals, color=cores, alpha=0.88, width=0.65)
for b, v in zip(bars, vals):
    ax1.text(b.get_x() + b.get_width() / 2, v + 0.6, pct(v / 100), ha='center', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 36)
ax1.set_ylabel('Pessoas ocupadas (%)', fontsize=12)
ax1.set_title(f'Em cada cem pessoas, {vir(br.p15 * 100, 0)} chegam ao trabalho em quinze minutos\n'
              f'Censo 2022 · {mil(br.total)} pessoas que saem para trabalhar e voltam no dia', fontsize=14, pad=16)
ax1.text(0.99, 0.95, 'verde: até 15 minutos · vermelho: mais de uma hora', transform=ax1.transAxes,
         ha='right', fontsize=10, color=CINZA)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-128-grafico-01-sete-faixas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · capital a capital ══
fig2, ax2 = plt.subplots(figsize=(12, 8))
cc = cap.sort_values('p15')
y = np.arange(len(cc))
cor = [DOURADO if l == 'Bauru (SP)' else (VERMELHO if l in ('São Paulo (SP)', 'Rio de Janeiro (RJ)') else VERDE) for l in cc.local]
ax2.barh(y, cc.p15 * 100, color=cor, alpha=0.88, height=0.65)
for i, v in enumerate(cc.p15 * 100):
    ax2.text(v + 0.4, i, pct(v / 100), va='center', fontsize=11)
ax2.axvline(br.p15 * 100, color=CINZA, linestyle='--', linewidth=2)
ax2.text(br.p15 * 100 + 0.4, len(cc) - 0.4, f'Brasil {pct(br.p15)}', color=CINZA, fontsize=11)
ax2.set_yticks(y); ax2.set_yticklabels(cc.local, fontsize=11)
ax2.set_xlim(0, 45)
ax2.set_xlabel('Chegam ao trabalho em até 15 minutos (%)', fontsize=12)
ax2.set_title('Em São Paulo, 15 em cada cem. Em Bauru, 38.\nDoze capitais e Bauru, Censo 2022', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-128-grafico-02-capital-a-capital.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · homens e mulheres ══
fig3, ax3 = plt.subplots(figsize=(12, 7))
x = np.arange(len(FAIXAS)); w = 0.38
vh = [h[f] / h.total * 100 for f in FAIXAS]; vm = [m[f] / m.total * 100 for f in FAIXAS]
ax3.bar(x - w / 2, vh, w, color=VERDE, alpha=0.85, label='homens')
ax3.bar(x + w / 2, vm, w, color=DOURADO, alpha=0.9, label='mulheres')
for i in range(len(FAIXAS)):
    ax3.text(x[i] - w / 2, vh[i] + 0.5, pct(vh[i] / 100), ha='center', fontsize=9.5, color=VERDE)
    ax3.text(x[i] + w / 2, vm[i] + 0.5, pct(vm[i] / 100), ha='center', fontsize=9.5, color='#8a6f1f')
ax3.set_xticks(x); ax3.set_xticklabels(ROTULOS, fontsize=11)
ax3.set_ylim(0, 36); ax3.set_ylabel('Pessoas ocupadas (%)', fontsize=12)
ax3.legend(frameon=False, fontsize=11)
ax3.set_title(f'As mulheres chegam mais em quinze minutos ({pct(m.p15)} contra {pct(h.p15)})\n'
              f'e a cauda das quatro horas é quase só de homens ({mil(h.mais_de_240)} contra {mil(m.mais_de_240)})',
              fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-128-grafico-03-homens-e-mulheres.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · uma sala de trinta ══
fig4, ax4 = plt.subplots(figsize=(12, 7))
bins = np.arange(-0.5, N_SALA + 1.5, 1)
for (nome, x_), cor_ in zip(sala.items(), [CINZA, VERMELHO, DOURADO]):
    ax4.hist(x_, bins=bins, density=True, alpha=0.55, color=cor_, label=f'{nome}: média {vir(x_.mean())}')
ax4.set_xlim(-0.5, 24.5)
ax4.set_xlabel(f'Colegas, de {N_SALA}, que chegaram ao trabalho em quinze minutos', fontsize=12)
ax4.set_ylabel('Fração das salas sorteadas', fontsize=12)
ax4.legend(frameon=False, fontsize=11)
ax4.set_title(f'Numa sala de {N_SALA}, quantos chegaram em quinze minutos?\n'
              f'{mil(SORTEIOS)} salas sorteadas por lugar, semente {SEMENTE}', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-128-grafico-04-uma-sala-de-trinta.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**A chance de você chegar ao trabalho em quinze minutos é de 36%.** Mais de uma em cada
três pessoas que saem de casa para trabalhar está lá em quinze minutos. Uma em cada dez
chega em cinco.

A faixa mais comum é a de quinze a trinta minutos. Mais de uma hora é a realidade de 12%,
uma em cada oito pessoas.

O lugar muda quase tudo. Em São Paulo, 15 em cada cem chegam em quinze minutos. No Rio,
15. Em Bauru, 38. Numa sala de trinta colegas sorteada em São Paulo, quatro ou cinco
chegaram em quinze minutos. Em Bauru, onze.

As mulheres chegam um pouco mais em quinze minutos do que os homens, 38% contra 34%. Na
outra ponta, a dos trajetos de mais de quatro horas, há seis homens para cada mulher em
número, e quatro vezes e meia mais em proporção.

Numa conta de bolso, com 220 dias de trabalho por ano, quem leva de uma a duas horas gasta
no caminho cerca de 660 horas por ano. Quem chega em quinze, cerca de 73.

*Quanto tempo você leva, e o que faz com ele?*

---

### ⚠️ Limitações do Modelo

- **O Censo pergunta o tempo habitual, em faixas.** Não há minutos exatos, e o dia ruim
  de trânsito não aparece. A conta de bolso usa o ponto médio de cada faixa, com a última
  fixada em 270 minutos, premissa nossa.
- **Só quem sai de casa e volta no dia.** Quem trabalha em casa, quem dorme no trabalho e
  quem não trabalha ficam fora. A pergunta do dia é sobre quem faz o trajeto.
- **Cerca de 1% não declarou o tempo.** As fatias são calculadas sobre o total, como o
  IBGE faz; sobre os declarados, cada fatia sobe um pouco.
- **O intervalo é o mínimo.** O Censo 2022 é amostra complexa e o IBGE não publicou o
  coeficiente de variação destas células. O intervalo de Jeffreys dá a largura mínima.
- **A sala de trinta é hipótese.** Colegas sorteados da cidade inteira. Na vida, quem
  trabalha junto mora em lugares parecidos, e a sala real fica mais extrema.
- **Capital não é região metropolitana.** Os números de São Paulo e do Rio são do
  município; quem mora fora e trabalha dentro conta na cidade onde mora.
- **A tabela de meio de transporte tem universo um pouco diferente** (71.296.180 contra
  71.227.656), e o cruzamento tempo × transporte não foi feito.
- **Os dados foram lidos pela API do SIDRA em 16/10/2026** e transcritos para os dois
  CSVs; os JSONs originais não foram salvos.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*